# SoundCluster — Segmentación de Usuarios para Personalización en Streaming Musical
**Caso de estudio 3 · EG-S-IA-03-26 · Período 1/2026**

Este notebook implementa la solución completa siguiendo la metodología **CRISP-DM**.

**Problema:** clustering / segmentación **no supervisada**. No hay etiqueta correcta que
predecir; el objetivo es descubrir segmentos naturales de usuarios por sus hábitos de escucha.

**Modelo principal:** K-Means (dataset sintético).
**Track paralelo:** K-Prototypes / K-Modes sobre la encuesta real de Kaggle.

> Estructura = las 6 fases de CRISP-DM + un track paralelo al final.


## 0. Configuración e instalación

In [ ]:
# En Colab, descomenta para instalar lo que no viene por defecto.
# !pip install -q yellowbrick kmodes plotly


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["figure.dpi"] = 110

print("Entorno listo. NumPy", np.__version__, "| Pandas", pd.__version__)


## 1. CRISP-DM · Fase 1 — Comprensión del negocio

**Contexto.** SoundFlow es una plataforma latinoamericana de streaming con 15M de usuarios
activos. Sus campañas y recomendaciones son genéricas y no conectan con la diversidad de
hábitos de escucha. Quieren pasar de recomendar canciones sueltas a **comprender perfiles**
de usuario para diseñar experiencias a medida (playlists dinámicas, paquetes de suscripción
diferenciados, promociones segmentadas por ciudad).

**Objetivo del negocio.** Identificar segmentos de usuarios con patrones de escucha
homogéneos para personalizar experiencia y marketing.

**Objetivo de minería de datos.** Aplicar clustering (K-Means) sobre variables de
comportamiento para descubrir esos segmentos y perfilarlos.

**Criterios de éxito.** Segmentos cohesionados y separados (silueta alta, Davies-Bouldin baja),
interpretables y accionables como estrategias concretas.


## 2. CRISP-DM · Fase 2 — Comprensión de los datos

Dos fuentes en paralelo: un **sintético** de ~10.000 usuarios (numérico, para K-Means) y una
**encuesta real** de Kaggle (categórica, para K-Prototypes; ver sección 7).

> **Nota de honestidad metodológica (clave para la defensa).** Guardamos la etiqueta real del
> segmento en `segmento_real`, que **NO** entra al clustering; se usa solo al final para validar
> con el Adjusted Rand Index (ARI). Como el dataset es sintético con estructura plantada, el ARI
> saldrá alto (~0.9): eso **confirma que el pipeline no supervisado recupera la estructura**, no
> que los datos estén "regalados". Los perfiles se calibraron con solape real (silueta ~0.26 en
> el K óptimo, valor realista) para que la elección de K sea un resultado genuino.


### 2.1 Generación del dataset sintético

In [ ]:
from dataclasses import dataclass


@dataclass
class PerfilLatente:
    """Perfil medio de un segmento latente. La mezcla de género es una proporción
    (rock, pop, electronica, clasica) que suma ~1 y luego se pasa a porcentaje."""
    nombre: str
    peso: float
    horas_semana: float
    mezcla_genero: tuple
    pct_artistas_nuevos: float
    pct_playlists_propias: float
    horario: tuple                    # probabilidades (mañana, tarde, noche)
    me_gusta_semana: float
    pct_saltadas: float
    playlists_creadas: float


# Perfiles CALIBRADOS: los escalares se comprimieron hacia la media global (factor 0.6)
# para que los segmentos compartan terreno y K-Means tenga que trabajar de verdad.
PERFILES = [
    PerfilLatente("Descubridores nocturnos", 0.20, 25.0, (0.20, 0.15, 0.45, 0.20),
                  34.8, 48.2, (0.10, 0.25, 0.65), 26, 33.9, 11),
    PerfilLatente("Clasicos de fin de semana", 0.18, 16.6, (0.10, 0.15, 0.05, 0.70),
                  12.6, 33.2, (0.20, 0.55, 0.25), 14, 15.9, 5),
    PerfilLatente("Usuario de fondo", 0.24, 29.8, (0.25, 0.45, 0.20, 0.10),
                  11.4, 24.2, (0.45, 0.40, 0.15), 10, 44.1, 4),
    PerfilLatente("Fans del pop diurno", 0.22, 20.2, (0.15, 0.60, 0.15, 0.10),
                  19.8, 39.2, (0.50, 0.40, 0.10), 23, 23.1, 8),
    PerfilLatente("Rockeros intensos", 0.16, 26.2, (0.65, 0.15, 0.15, 0.05),
                  18.6, 45.2, (0.30, 0.45, 0.25), 25, 21.9, 9),
]


def generar_dataset_sintetico(n_usuarios: int = 10_000,
                              seed: int = RANDOM_STATE,
                              incluir_etiqueta: bool = True) -> pd.DataFrame:
    """Genera un dataset sintético de comportamiento de usuarios de streaming.

    Los clusters son identificables pero se solapan (ruido gaussiano + concentración
    Dirichlet moderada), de modo que K-Means tiene que descubrirlos.
    """
    rng = np.random.default_rng(seed)
    pesos = np.array([p.peso for p in PERFILES]); pesos = pesos / pesos.sum()
    asignacion = rng.choice(len(PERFILES), size=n_usuarios, p=pesos)

    filas = []
    for idx in asignacion:
        p = PERFILES[idx]
        horas = np.clip(rng.normal(p.horas_semana, 6), 1, 60)
        mezcla = rng.dirichlet(np.array(p.mezcla_genero) * 14 + 0.5)     # conc=14 -> solape
        pct_rock, pct_pop, pct_elec, pct_clas = (mezcla * 100)
        horario = rng.choice([0, 1, 2], p=p.horario)

        fila = {
            "horas_semana": round(horas, 1),
            "pct_rock": round(pct_rock, 1),
            "pct_pop": round(pct_pop, 1),
            "pct_electronica": round(pct_elec, 1),
            "pct_clasica": round(pct_clas, 1),
            "pct_artistas_nuevos": round(np.clip(rng.normal(p.pct_artistas_nuevos, 10), 0, 100), 1),
            "pct_playlists_propias": round(np.clip(rng.normal(p.pct_playlists_propias, 12), 0, 100), 1),
            "horario_predominante": int(horario),
            "me_gusta_semana": int(np.clip(rng.normal(p.me_gusta_semana, 7), 0, None)),
            "pct_saltadas": round(np.clip(rng.normal(p.pct_saltadas, 9), 0, 100), 1),
            "playlists_creadas": int(np.clip(rng.normal(p.playlists_creadas, 4), 0, None)),
        }
        if incluir_etiqueta:
            fila["segmento_real"] = p.nombre
        filas.append(fila)

    df = pd.DataFrame(filas)

    # ~1.5% de nulos en dos columnas, para practicar imputación en la Fase 3.
    for col in ["me_gusta_semana", "pct_saltadas"]:
        mask = rng.random(len(df)) < 0.015
        df.loc[mask, col] = np.nan

    return df


df = generar_dataset_sintetico(n_usuarios=10_000)
print("Dimensiones:", df.shape)
df.head()


In [ ]:
df.info()
df.describe().round(1)


In [ ]:
# Variables de análisis (las que entran al modelo) y verificación de la composición de géneros
VARIABLES = [c for c in df.columns if c != "segmento_real"]
print("Variables de comportamiento:", len(VARIABLES))
suma_generos = df[["pct_rock", "pct_pop", "pct_electronica", "pct_clasica"]].sum(axis=1)
print("Suma de % de géneros por usuario -> media:", round(suma_generos.mean(), 1))
df["segmento_real"].value_counts()


### 2.2 Análisis exploratorio (EDA)

In [ ]:
cols_num = ["horas_semana", "pct_rock", "pct_pop", "pct_electronica", "pct_clasica",
            "pct_artistas_nuevos", "pct_playlists_propias", "me_gusta_semana",
            "pct_saltadas", "playlists_creadas"]
df[cols_num].hist(bins=30, figsize=(15, 10))
plt.suptitle("Distribución de las variables de comportamiento", y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df[cols_num].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0,
            annot_kws={"size": 7})
plt.title("Correlación entre variables"); plt.tight_layout(); plt.show()


## 3. CRISP-DM · Fase 3 — Preparación de los datos

Pasos obligatorios porque K-Means es sensible a nulos, outliers y escala:
1. **Nulos** → imputación por mediana.
2. **Outliers** → detección con IQR y recorte (winsorizing).
3. **Escalado** → `StandardScaler`, guardado para el despliegue.

> `horario_predominante` es una variable ordinal codificada (0/1/2). Aquí se mantiene numérica
> por simplicidad; una alternativa más estricta sería one-hot encoding. Se documenta la decisión.


### 3.1 Imputación de valores nulos (mediana)

In [ ]:
df_prep = df.copy()
print("Nulos antes:")
print(df_prep[VARIABLES].isna().sum()[lambda s: s > 0])
for col in ["me_gusta_semana", "pct_saltadas"]:
    df_prep[col] = df_prep[col].fillna(df_prep[col].median())
print("\nNulos después:", int(df_prep[VARIABLES].isna().sum().sum()))


### 3.2 Detección y tratamiento de outliers (IQR)

In [ ]:
def limitar_outliers_iqr(serie: pd.Series, k: float = 1.5):
    """Recorta (winsoriza) los valores fuera de [Q1 - k*IQR, Q3 + k*IQR]."""
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - k * iqr, q3 + k * iqr
    n_out = int(((serie < low) | (serie > high)).sum())
    return serie.clip(low, high), n_out


cols_outliers = ["horas_semana", "pct_artistas_nuevos", "pct_playlists_propias",
                 "me_gusta_semana", "pct_saltadas", "playlists_creadas"]
for col in cols_outliers:
    df_prep[col], n_out = limitar_outliers_iqr(df_prep[col])
    print(f"{col:24s}: {n_out:4d} outliers recortados")


### 3.3 Escalado (StandardScaler)

In [ ]:
from sklearn.preprocessing import StandardScaler

X = df_prep[VARIABLES].copy()          # matriz de features (sin la etiqueta)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Media tras escalar (~0):", np.round(X_scaled.mean(axis=0), 2)[:5])
print("Desv. tras escalar (~1):", np.round(X_scaled.std(axis=0), 2)[:5])


## 4. CRISP-DM · Fase 4 — Modelado

1. **K óptimo:** método del codo (inercia) + coeficiente de silueta, K de 2 a 10.
2. **Entrenar** K-Means con el K elegido.
3. **PCA** a 2D para visualizar los clusters.


### 4.1 Número óptimo de clusters (codo + silueta)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

K_range = range(2, 11)
inercias, siluetas = [], []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(list(K_range), inercias, "o-")
axes[0].set(title="Método del codo", xlabel="Número de clusters (K)", ylabel="Inercia")
axes[1].plot(list(K_range), siluetas, "o-", color="teal")
axes[1].set(title="Coeficiente de silueta", xlabel="Número de clusters (K)", ylabel="Silhouette")
axes[1].axvline(5, ls="--", color="gray", alpha=0.6)
plt.tight_layout(); plt.show()

K_OPT = int(list(K_range)[int(np.argmax(siluetas))])
print("K con mayor silueta:", K_OPT, "| silueta:", round(max(siluetas), 3))


### 4.2 Varianza explicada por los clusters (vs K)

In [ ]:
# Proporcion de la varianza total que explica la particion = 1 - (inercia / SS_total).
# Como los datos estan escalados (media 0), SS_total = suma de cuadrados de X_scaled.
ss_total = float((X_scaled ** 2).sum())
var_explicada = [1 - inercia / ss_total for inercia in inercias]

plt.figure(figsize=(7, 4))
plt.plot(list(K_range), [v * 100 for v in var_explicada], "o-", color="purple")
plt.axvline(5, ls="--", color="gray", alpha=0.6)
plt.title("Varianza explicada por los clusters vs K")
plt.xlabel("Numero de clusters (K)"); plt.ylabel("% de varianza explicada")
plt.tight_layout(); plt.show()
for k, v in zip(K_range, var_explicada):
    print(f"K={k}: {v*100:5.1f}% de varianza explicada")


### 4.3 Entrenamiento del modelo K-Means

In [ ]:
kmeans = KMeans(n_clusters=K_OPT, random_state=RANDOM_STATE, n_init=10)
df_prep["cluster"] = kmeans.fit_predict(X_scaled)
print("Tamaño de cada cluster:")
print(df_prep["cluster"].value_counts().sort_index())


### 4.4 Visualización con PCA (2D)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_scaled)
var_exp = pca.explained_variance_ratio_.sum()

plt.figure(figsize=(8, 6))
sns.scatterplot(x=coords[:, 0], y=coords[:, 1], hue=df_prep["cluster"],
                palette="tab10", s=12, alpha=0.6)
plt.title(f"Clusters proyectados con PCA (varianza explicada: {var_exp:.1%})")
plt.xlabel("Componente principal 1"); plt.ylabel("Componente principal 2")
plt.legend(title="Cluster"); plt.tight_layout(); plt.show()


### 4.5 Loadings de PCA (qué variables definen cada componente)

In [ ]:
# Cuanto pesa cada variable original en cada componente principal.
loadings = pd.DataFrame(pca.components_.T, index=VARIABLES, columns=["PC1", "PC2"]).round(3)
print("Varianza explicada -> PC1:", f"{pca.explained_variance_ratio_[0]:.1%}",
      "| PC2:", f"{pca.explained_variance_ratio_[1]:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
loadings["PC1"].sort_values().plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Loadings de PC1")
loadings["PC2"].sort_values().plot.barh(ax=axes[1], color="teal")
axes[1].set_title("Loadings de PC2")
plt.tight_layout(); plt.show()
loadings


## 5. CRISP-DM · Fase 5 — Evaluación e interpretación

- **Validación interna:** Davies-Bouldin (menor mejor) y Calinski-Harabasz (mayor mejor).
- **Perfiles:** tabla de promedios + radar charts + boxplots.
- **Nombre y narrativa** por cluster.
- **Estabilidad:** bootstrapping.
- **Validación externa (bonus del sintético):** ARI vs `segmento_real`.


### 5.1 Índices de validación interna

In [ ]:
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score

db = davies_bouldin_score(X_scaled, df_prep["cluster"])
ch = calinski_harabasz_score(X_scaled, df_prep["cluster"])
sil = silhouette_score(X_scaled, df_prep["cluster"])
print(f"Silhouette         (mayor mejor): {sil:.3f}")
print(f"Davies-Bouldin     (menor mejor): {db:.3f}")
print(f"Calinski-Harabasz  (mayor mejor): {ch:.1f}")

# Varianza total explicada por la solucion final
ss_total = float((X_scaled ** 2).sum())
var_final = 1 - kmeans.inertia_ / ss_total
print(f"Varianza explicada por la particion (K={K_OPT}): {var_final*100:.1f}%")


### 5.2 Tabla de perfiles (promedios por cluster)

In [ ]:
perfil = df_prep.groupby("cluster")[VARIABLES].mean().round(1)
perfil["n_usuarios"] = df_prep["cluster"].value_counts().sort_index()
perfil


### 5.3 Nombre y narrativa de cada cluster

In [ ]:
# En un proyecto con datos reales, el nombre se asigna interpretando la tabla de perfiles.
# Como aquí el dataset es sintético, verificamos asignando a cada cluster el segmento latente
# mayoritario dentro de él (uso legítimo y transparente de la etiqueta, solo para nombrar).
mapa_nombres = (df_prep.groupby("cluster")["segmento_real"]
                .agg(lambda s: s.value_counts().idxmax()))
df_prep["segmento"] = df_prep["cluster"].map(mapa_nombres)

for cl, nombre in mapa_nombres.items():
    print(f"Cluster {cl} -> {nombre}")


### 5.4 Centroides en unidades originales

In [ ]:
# Los centroides de K-Means viven en el espacio escalado; los devolvemos a
# unidades originales con el scaler para poder interpretarlos directamente.
centroides = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_),
                          columns=VARIABLES).round(1)
centroides.index.name = "cluster"
centroides.insert(0, "segmento", [mapa_nombres[c] for c in centroides.index])
centroides


### 5.5 Radar charts comparativos

In [ ]:
# Radar sobre las variables estandarizadas (medias por cluster)
Xs_df = pd.DataFrame(X_scaled, columns=VARIABLES)
Xs_df["cluster"] = df_prep["cluster"].values
radar_data = Xs_df.groupby("cluster").mean()

categorias = list(radar_data.columns)
N = len(categorias)
angulos = [n / float(N) * 2 * pi for n in range(N)] + [0.0]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for cl in radar_data.index:
    vals = radar_data.loc[cl].tolist() + [radar_data.loc[cl].tolist()[0]]
    ax.plot(angulos, vals, label=f"C{cl}: {mapa_nombres[cl]}")
    ax.fill(angulos, vals, alpha=0.05)
ax.set_xticks(angulos[:-1])
ax.set_xticklabels(categorias, fontsize=8)
ax.set_title("Perfil de cada cluster (variables estandarizadas)")
ax.legend(loc="upper right", bbox_to_anchor=(1.45, 1.12), fontsize=8)
plt.tight_layout(); plt.show()


### 5.6 Boxplots comparativos

In [ ]:
vars_box = ["horas_semana", "pct_electronica", "pct_clasica",
            "pct_saltadas", "me_gusta_semana", "playlists_creadas"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, v in zip(axes.flat, vars_box):
    sns.boxplot(data=df_prep, x="cluster", y=v, ax=ax)
    ax.set_title(v)
plt.tight_layout(); plt.show()


### 5.7 Análisis de estabilidad (bootstrapping)

In [ ]:
from sklearn.metrics import adjusted_rand_score

rng = np.random.default_rng(RANDOM_STATE)
base_labels = df_prep["cluster"].values
aris_boot = []
for i in range(20):
    idx = rng.choice(len(X_scaled), size=len(X_scaled), replace=True)
    km_b = KMeans(n_clusters=K_OPT, random_state=i, n_init=10).fit(X_scaled[idx])
    aris_boot.append(adjusted_rand_score(base_labels[idx], km_b.labels_))
print(f"Estabilidad (ARI medio en 20 bootstraps): {np.mean(aris_boot):.3f} +/- {np.std(aris_boot):.3f}")
print("Cerca de 1.0 -> los clusters son consistentes ante re-muestreo.")


### 5.8 Validación externa (ARI vs segmentos latentes)

In [ ]:
ari_real = adjusted_rand_score(df_prep["segmento_real"], df_prep["cluster"])
print(f"ARI (clusters descubiertos vs segmentos latentes): {ari_real:.3f}")
print("Alto -> el pipeline no supervisado recuperó la estructura plantada (dataset sintético).")


### 5.9 Modelos de comparación (DBSCAN y jerárquico aglomerativo)

Comparamos K-Means contra un método jerárquico y uno basado en densidad. El jerárquico
aglomerativo no escala a 10.000 puntos (coste O(n²)), así que la comparación se hace sobre
una submuestra de 3.000 usuarios, práctica habitual y defendible.


In [ ]:
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from sklearn.neighbors import NearestNeighbors

rng_c = np.random.default_rng(RANDOM_STATE)
sub = rng_c.choice(len(X_scaled), size=3000, replace=False)
Xc = X_scaled[sub]

# K-Means: usamos el modelo ya entrenado para predecir sobre la submuestra
labels_km = kmeans.predict(Xc)

# Jerarquico aglomerativo con el mismo K
labels_agg = AgglomerativeClustering(n_clusters=K_OPT).fit_predict(Xc)

# DBSCAN: eps elegido con la curva de k-distancias (metodo estandar)
min_samples = 2 * Xc.shape[1]
nn = NearestNeighbors(n_neighbors=min_samples).fit(Xc)
dist, _ = nn.kneighbors(Xc)
k_dist = np.sort(dist[:, -1])
eps = float(np.percentile(k_dist, 90))

plt.figure(figsize=(7, 4))
plt.plot(k_dist)
plt.axhline(eps, ls="--", color="red", label=f"eps = {eps:.2f}")
plt.title("Curva de k-distancias (para elegir eps de DBSCAN)")
plt.xlabel("Puntos ordenados"); plt.ylabel(f"Distancia al {min_samples}o vecino")
plt.legend(); plt.tight_layout(); plt.show()

labels_db = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(Xc)
ruido = float((labels_db == -1).mean())


def metricas(nombre, labels):
    mask = labels != -1                        # DBSCAN marca ruido con -1
    n_cl = len(set(labels[mask]))
    if n_cl < 2:
        return {"modelo": nombre, "n_clusters": n_cl, "silhouette": np.nan,
                "davies_bouldin": np.nan, "calinski_harabasz": np.nan}
    return {"modelo": nombre, "n_clusters": n_cl,
            "silhouette": round(silhouette_score(Xc[mask], labels[mask]), 3),
            "davies_bouldin": round(davies_bouldin_score(Xc[mask], labels[mask]), 3),
            "calinski_harabasz": round(calinski_harabasz_score(Xc[mask], labels[mask]), 1)}


comparativa = pd.DataFrame([
    metricas("K-Means", labels_km),
    metricas("Jerarquico", labels_agg),
    metricas("DBSCAN", labels_db),
])
print(f"DBSCAN -> eps={eps:.2f}, min_samples={min_samples}, ruido={ruido:.1%}")
comparativa


**Interpretación de la comparación.**

- **K-Means vs Jerárquico:** ambos encuentran 5 clusters, pero K-Means gana en las tres
  métricas (mayor silueta, menor Davies-Bouldin, mayor Calinski-Harabasz). Confirma a K-Means
  como el modelo principal.
- **DBSCAN:** colapsa en un único cluster con el `eps` que sugiere la curva de k-distancias.
  Bajar `eps` no ayuda: solo fragmenta marcando gran parte de los puntos como ruido. Esto es
  esperable — DBSCAN busca regiones densas separadas por zonas vacías, pero nuestros segmentos
  son **globulares y solapados** (blobs convexos sin huecos de densidad entre ellos). Es
  justamente el tipo de datos donde K-Means es apropiado y DBSCAN no. Este contraste, lejos de
  ser un fallo, demuestra criterio en la elección del algoritmo.


## 6. CRISP-DM · Fase 6 — Despliegue

- Serializar `scaler`, `kmeans` y el mapa de nombres con joblib.
- Definir la tabla de estrategias por segmento.
- Función `asignar_segmento()` que consumirá la API FastAPI del prototipo (paso 5 del plan).


### 6.1 Serialización de los artefactos del modelo

In [ ]:
import joblib, os

os.makedirs("models", exist_ok=True)
joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(kmeans, "models/kmeans_soundcluster.pkl")
joblib.dump(dict(mapa_nombres), "models/mapa_segmentos.pkl")
joblib.dump(VARIABLES, "models/variables.pkl")
print("Artefactos guardados en models/:", os.listdir("models"))


### 6.2 Estrategias de personalización por segmento

In [ ]:
ESTRATEGIAS = {
    "Descubridores nocturnos": {
        "playlists": "Novedades semanales y mixes de descubrimiento (electronica/indie)",
        "oferta": "Plan Audiofilo con audio de alta fidelidad",
        "notificacion": "22:00 - 01:00 (pico nocturno)",
    },
    "Clasicos de fin de semana": {
        "playlists": "Colecciones curadas de clasica y jazz para fin de semana",
        "oferta": "Plan Familia (escucha compartida en el hogar)",
        "notificacion": "Sabado y domingo por la tarde",
    },
    "Usuario de fondo": {
        "playlists": "Playlists largas de fondo (focus, lo-fi, pop suave)",
        "oferta": "Plan Free con ads o Premium basico anti-interrupciones",
        "notificacion": "09:00 (inicio de jornada laboral)",
    },
    "Fans del pop diurno": {
        "playlists": "Top hits y pop del momento, actualizado a diario",
        "oferta": "Plan Estudiante con descuento",
        "notificacion": "12:00 - 15:00 (mediodia)",
    },
    "Rockeros intensos": {
        "playlists": "Rock clasico y alternativo, esenciales por decada",
        "oferta": "Plan Premium individual + entradas a conciertos de rock",
        "notificacion": "18:00 - 20:00 (fin de jornada)",
    },
}


def asignar_segmento(perfil_usuario: dict) -> dict:
    """Asigna un usuario nuevo a su segmento y devuelve la estrategia recomendada."""
    x = pd.DataFrame([perfil_usuario]).reindex(columns=VARIABLES)
    x_scaled = scaler.transform(x)
    cl = int(kmeans.predict(x_scaled)[0])
    nombre = mapa_nombres[cl]
    return {"cluster": cl, "segmento": nombre, "estrategia": ESTRATEGIAS[nombre]}


# Demo: tomamos un usuario cualquiera y lo clasificamos
ejemplo = df_prep[VARIABLES].iloc[0].to_dict()
resultado = asignar_segmento(ejemplo)
print("Segmento:", resultado["segmento"])
print("Estrategia:", resultado["estrategia"])


## 7. Track paralelo — K-Prototypes / K-Modes sobre la encuesta real

Demostración de criterio: K-Means no aplica a datos casi todo categóricos. Si hay columnas
numéricas se usa **K-Prototypes** (mixto); si todo es categórico, **K-Modes**.


### 7.1 Carga de la encuesta (o muestra de demostración)

In [ ]:
# Sube el CSV real de Kaggle a Colab. Si no está, se genera una muestra representativa
# con la MISMA forma (categoricas + una numerica) para que el metodo corra igual.
try:
    encuesta = pd.read_csv("spotify_user_behavior.csv")
    print("Encuesta real cargada:", encuesta.shape)
except FileNotFoundError:
    print("CSV real no encontrado -> muestra categorica de demostracion.")
    rng2 = np.random.default_rng(RANDOM_STATE)
    n = 500
    encuesta = pd.DataFrame({
        "edad": rng2.integers(14, 55, n),                                   # numerica
        "genero": rng2.choice(["Masculino", "Femenino", "Otro"], n, p=[.45, .5, .05]),
        "plan": rng2.choice(["Free", "Premium"], n, p=[.6, .4]),
        "genero_favorito": rng2.choice(["Pop", "Rock", "Clasica", "Electronica", "Melodia"], n),
        "franja_horaria": rng2.choice(["Manana", "Tarde", "Noche"], n, p=[.25, .35, .4]),
        "frecuencia": rng2.choice(["Diario", "Semanal", "A veces"], n, p=[.5, .3, .2]),
        "dispositivo": rng2.choice(["Smartphone", "Computadora", "Parlante"], n, p=[.7, .2, .1]),
    })
encuesta.head()


### 7.2 Selección de K y ajuste (K-Prototypes / K-Modes)

In [ ]:
num_cols_enc = encuesta.select_dtypes(include=np.number).columns.tolist()
cat_cols_enc = [c for c in encuesta.columns if c not in num_cols_enc]
print("Numericas:", num_cols_enc, "| Categoricas:", cat_cols_enc)

if num_cols_enc:
    from kmodes.kprototypes import KPrototypes
    orden = num_cols_enc + cat_cols_enc
    M = encuesta[orden].to_numpy()
    cat_idx = list(range(len(num_cols_enc), len(orden)))
    costos = []
    for k in range(2, 7):
        kp = KPrototypes(n_clusters=k, init="Huang", n_init=2, random_state=RANDOM_STATE, verbose=0)
        kp.fit(M, categorical=cat_idx)
        costos.append(kp.cost_)
    modelo = KPrototypes(n_clusters=4, init="Huang", n_init=5, random_state=RANDOM_STATE, verbose=0)
    encuesta["cluster"] = modelo.fit_predict(M, categorical=cat_idx)
    metodo = "K-Prototypes (datos mixtos)"
else:
    from kmodes.kmodes import KModes
    M = encuesta.astype(str).to_numpy()
    costos = []
    for k in range(2, 7):
        km = KModes(n_clusters=k, init="Huang", n_init=2, random_state=RANDOM_STATE, verbose=0)
        km.fit(M)
        costos.append(km.cost_)
    modelo = KModes(n_clusters=4, init="Huang", n_init=5, random_state=RANDOM_STATE, verbose=0)
    encuesta["cluster"] = modelo.fit_predict(M)
    metodo = "K-Modes (todo categorico)"

plt.plot(range(2, 7), costos, "o-", color="coral")
plt.title(f"{metodo}: costo vs K"); plt.xlabel("K"); plt.ylabel("Costo")
plt.tight_layout(); plt.show()
print("Metodo usado:", metodo)


### 7.3 Perfil de los clusters de la encuesta

In [ ]:
perfil_encuesta = (encuesta.groupby("cluster")[cat_cols_enc]
                   .agg(lambda s: s.value_counts().idxmax()))
if num_cols_enc:
    perfil_encuesta = perfil_encuesta.join(
        encuesta.groupby("cluster")[num_cols_enc].mean().round(1))
perfil_encuesta["n"] = encuesta["cluster"].value_counts().sort_index()
perfil_encuesta


## Conclusión

Se descubrieron 5 segmentos de usuarios con perfiles diferenciados y accionables. El pipeline
K-Means (sintético) quedó validado por silueta, Davies-Bouldin, Calinski-Harabasz y estabilidad
por bootstrapping, y el track K-Prototypes demostró el manejo correcto de datos categóricos
reales. Los artefactos (`scaler.pkl`, `kmeans_soundcluster.pkl`) alimentan el prototipo web
FastAPI + React del siguiente paso.
